# Ứng dụng 2: Mẫu trực quan trên ảnh (Caltech-101)

**Câu hỏi:** mỗi loại vật thể có những tổ hợp chi tiết hình ảnh nào đặc trưng, và các tổ hợp đó có giúp phân loại ảnh không?

Notebook này chỉ **điều phối**: mỗi bước là một file trong `applications/app2_image/src/`, ghi kết quả trung gian ra `applications/app2_image/data/` và `applications/app2_image/out/`.
Nếu Colab ngắt giữa chừng, chạy lại cell thiết lập rồi chạy tiếp từ bước đang dở (các bước bỏ qua phần đã làm xong).

| Bước | File | Kết quả |
|---|---|---|
| 1 | `s01_prepare_data.py` | `data/images/`, `data/split.csv` |
| 1b | `s02_sift_probe.py` | `out/sift_probe.csv`, `out/sift_probe.png` |
| 2 | `s03_sift_extract.py` | `data/sift.npz`, `out/sift_stats.csv` |
| 3 | `s04_codebook.py` | `data/codebook_k{100,300}.npy`, `data/words.npz`, `out/codebook_stats.csv` |
| 4 | `s05_transactions.py` | `data/transactions/k{100,300}_{word,grid}/`, `out/transaction_stats.csv`, `out/tau_sensitivity.csv` |
| 5–7 | `s06` … `s08` | *(đang làm)* |

## Thư viện sử dụng

Theo hướng dẫn của thầy, phần tiền xử lý được dùng thư viện hỗ trợ. **Khai thác tập phổ biến chỉ dùng FPClose do nhóm tự cài đặt (Julia)**; không dùng mlxtend, efficient-apriori hay thư viện FIM nào khác. SPMF chỉ dùng để đối chiếu kết quả.

**Tiền xử lý (biến ảnh thành giao dịch)**

| Thư viện | Dùng để làm gì | Bước |
|---|---|---|
| OpenCV (`opencv-python-headless`) | Đọc ảnh, chuyển xám, resize; trích đặc trưng SIFT | 1, 2 |
| NumPy | Mảng descriptor, tọa độ keypoint, histogram | 2–4 |
| scikit-learn | Chia train/test theo lớp (`train_test_split`); gom cụm tạo codebook (`MiniBatchKMeans`); đo accuracy CV khi chọn τ (`StratifiedKFold`, `cross_val_score`, `LogisticRegression`) | 1, 3, 4 |
| pandas | Danh sách ảnh, bảng thống kê | 1–4 |

**Khai thác:** FPClose của nhóm (`demo.jl`), bước 5.

**Phân tích và đánh giá sau khai thác**

| Thư viện | Dùng để làm gì | Bước |
|---|---|---|
| pandas | Bảng mẫu, growth rate, bảng kết quả | 6, 7 |
| Matplotlib | Vẽ keypoint và mẫu lên ảnh, biểu đồ | 1b, 6, 7 |
| scikit-learn | Phân loại (`LogisticRegression`), accuracy, macro-F1, confusion matrix | 7 |


## 0. Thiết lập môi trường

In [ ]:
import os, sys, subprocess
from pathlib import Path

USE_OWN_IMPL = True      # True: khai thác bằng FPClose (Julia) của nhóm. Bản nộp PHẢI là True.
REPO_URL = "https://github.com/DRAGON-1312/FPClose.git"
REPO_BRANCH = "main"     # muốn thử trên Colab trước khi merge: đổi thành nhánh đang làm, vd "feature/app2"
APP_REL = Path("applications/app2_image")
IN_COLAB = "google.colab" in sys.modules

def find_repo_dir():
    """Gốc repo FPClose: đi từ thư mục hiện tại lên trên, tìm thư mục có applications/app2_image."""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / APP_REL / "src" / "s01_prepare_data.py").exists():
            return d
    return None

REPO_DIR = find_repo_dir()
if REPO_DIR is None and IN_COLAB:                  # Colab: clone repo về /content/FPClose (một lần)
    REPO_DIR = Path("/content/FPClose")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
assert REPO_DIR is not None and (REPO_DIR / APP_REL / "src").is_dir(), \
    "Không tìm thấy applications/app2_image: mở notebook từ trong repo FPClose, hoặc chạy trên Colab"
APP_DIR = REPO_DIR / APP_REL

os.chdir(APP_DIR)                                  # các bước dùng đường dẫn tương đối src/, data/, out/
os.environ["PYTHONIOENCODING"] = "utf-8"           # để in tiếng Việt trên console Windows
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(APP_DIR / "src") not in sys.path:
    sys.path.insert(0, str(APP_DIR / "src"))

def run_step(script, *args):
    """Chạy một bước trong src/ bằng đúng Python của kernel, in output, dừng nếu lỗi."""
    r = subprocess.run([sys.executable, f"src/{script}", *map(str, args)],
                       capture_output=True, text=True, encoding="utf-8")
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{script} lỗi (mã {r.returncode})")

import config
print("REPO_DIR =", REPO_DIR, "| Colab:", IN_COLAB, "| SEED =", config.SEED)

## 1. Chuẩn bị dữ liệu

- **5 lớp** `airplanes`, `Motorbikes`, `Faces_easy`, `watch`, `Leopards`: hình dạng khác nhau rõ, đủ ảnh. Bỏ `Faces` (trùng `Faces_easy`) và `BACKGROUND_Google` (không phải một loại vật thể).
- **200 ảnh/lớp**: `Leopards` chỉ có 200 ảnh, nên đây là mức cân bằng lớn nhất. Lớp cân bằng thì support giữa các lớp mới so sánh được (growth rate).
- **Train/test 70/30 theo từng lớp** (140/60), seed 42. Mẫu chỉ khai thác trên train.
- **Ảnh xám, cạnh dài 400 px**: SIFT chỉ dùng độ sáng; cùng kích thước để số keypoint không phụ thuộc độ phân giải gốc.

In [ ]:
run_step("s01_prepare_data.py")

In [ ]:
import pandas as pd
split = pd.read_csv(config.DATA_SPLIT)
pd.crosstab(split.label, split.split, margins=True)

## 1b. Chạy thử SIFT

`cv2.SIFT_create(nfeatures=500)` trên 10 ảnh train mỗi lớp, để xem số keypoint trung bình trước khi chạy toàn bộ.

In [ ]:
run_step("s02_sift_probe.py", 10)

In [ ]:
from IPython.display import Image, display
probe = pd.read_csv(config.OUT / "sift_probe.csv")
display(probe.groupby("label").n_kp.agg(["count", "mean", "min", "max"]).round(1))
Image(str(config.OUT / "sift_probe.png"))

**Nhận xét (lần chạy thử):**

- Trung bình ~437 keypoint/ảnh, descriptor 128 chiều → 700 ảnh train cho ~300k descriptor, lấy mẫu 150k cho k-means là đủ.
- `airplanes` ít keypoint hơn hẳn (~230) vì nền trời trơn → giao dịch ngắn hơn, cần lưu ý khi chọn minsup theo lớp.
- Hầu hết các lớp chạm trần 500 → giữ `nfeatures=500` để độ dài giao dịch giữa các lớp tương đương.
- **Thiên lệch nền:** `Faces_easy` có nhiều keypoint trên kệ sách phía sau, `Leopards` có cỏ và viền đen. Sẽ phân tích ở bước vẽ mẫu.

## 2. Trích SIFT toàn bộ ảnh

Chạy `cv2.SIFT_create(nfeatures=500)` trên cả 1000 ảnh (train và test), lưu **descriptor** (128 chiều) và **tọa độ keypoint** (x, y, size, angle) vào `data/sift.npz`.

- **Vì sao lưu cả test:** ở bước 3 mọi keypoint (train và test) đều được gán từ thị giác bằng codebook; codebook chỉ học từ train nên không rò rỉ.
- **Vì sao lưu tọa độ:** bước vẽ mẫu lên ảnh gốc và mã hóa vị trí lưới 2×2 cần biết keypoint nằm ở đâu.
- **Cách lưu:** một mảng phẳng + `offsets` (keypoint của ảnh *i* nằm ở `[offsets[i], offsets[i+1])`), thứ tự ảnh = thứ tự dòng của `data/split.csv`. Descriptor của OpenCV vốn là số nguyên 0–255 nên lưu `uint8` không mất thông tin (nhỏ hơn 4 lần, file ~49 MB).
- SIFT là thuật toán xác định, không có bước ngẫu nhiên nên chạy lại cho kết quả giống hệt.

In [ ]:
run_step("s03_sift_extract.py")

In [ ]:
import numpy as np
sift_stats = pd.read_csv(config.OUT / "sift_stats.csv")
z = np.load(config.DATA_SIFT)
is_train = (sift_stats.split == "train").to_numpy()
n_kp = np.diff(z["offsets"])
print(f"Descriptor: {z['desc'].shape}, train {n_kp[is_train].sum():,} / test {n_kp[~is_train].sum():,} keypoint")
sift_stats.pivot_table(index="label", columns="split", values="n_kp", aggfunc="mean").round(1).loc[config.CLASSES]

**Nhận xét:**

- 1000 ảnh cho ~450k keypoint, trong đó **~316k descriptor thuộc train** → bước 3 lấy mẫu khoảng 150k descriptor (một nửa) cho k-means là hợp lý, đủ phủ mà không tốn thời gian.
- Khác lần chạy thử 10 ảnh (airplanes ~230), toàn bộ train cho airplanes trung bình ~282 keypoint/ảnh; vẫn thấp hơn hẳn 4 lớp còn lại (~480–500, gần chạm trần 500). Giao dịch của airplanes sẽ **ngắn hơn** → cùng một minsup phần trăm nhưng mật độ khác, cần xem ở bảng thống kê bước 4.
- Không ảnh nào thiếu keypoint (ít nhất 75 keypoint), nên không phải loại ảnh nào.

## 3. Codebook và gán từ thị giác

Gom các descriptor SIFT thành *k* cụm bằng `MiniBatchKMeans`; tâm mỗi cụm là một **từ thị giác** (một kiểu mảnh ảnh cục bộ). Mỗi keypoint được gán từ của tâm gần nhất. Một ảnh khi đó là một "túi từ", và là một giao dịch ở bước sau.

- **Chỉ học từ train.** Mẫu lấy từ 700 ảnh train; test chỉ được *gán từ* bằng codebook đã học, không tham gia học. Nếu test lọt vào k-means thì thông tin test rò sang bước phân loại.
- **~150k descriptor, đều theo ảnh:** mỗi ảnh train đóng góp tối đa *q* descriptor (*q* tìm tự động, ra 221), ảnh ít hơn *q* thì lấy hết. Lấy ngẫu nhiên thuần túy sẽ để Motorbikes/Leopards (~500 keypoint/ảnh) chiếm phần lớn, còn airplanes (~282) bị lép, nên codebook sẽ thiên về các lớp nhiều keypoint. 150k là gần một nửa số descriptor train, đủ phủ mà k-means vẫn chạy trong vài giây.
- **k = 100 và 300**, so sánh cả hai ở phần phân loại:
  - k nhỏ: mỗi từ thô, xuất hiện trong nhiều ảnh → support cao, giao dịch "dày", FPClose cho ra nhiều mẫu nhưng kém đặc trưng.
  - k lớn: từ chi tiết hơn, phân biệt lớp tốt hơn nhưng support thấp hơn → phải hạ minsup, giao dịch thưa.
- **`MiniBatchKMeans`** thay vì `KMeans`: nhanh hơn nhiều với dữ liệu lớn. Đã kiểm tra: inertia chỉ cao hơn `KMeans` đầy đủ ~1,8% (k=100 và k=300) nên chất lượng gần như tương đương. `n_init=3`, `batch_size=4096`, `random_state=42`; chạy lại cho codebook giống hệt (đã so hash file).
- Từ được đánh số 0…k−1 ở bước này; bước giao dịch mới đổi sang item SPMF bắt đầu từ 1.

In [ ]:
run_step("s04_codebook.py")

In [ ]:
words = np.load(config.DATA_WORDS)
print({name: (w.shape, w.dtype) for name, w in words.items()}, "| khớp số keypoint của sift.npz:", all(len(w) == len(z["x"]) for w in words.values()))
pd.read_csv(config.OUT / "codebook_stats.csv")

**Nhận xét:**

- Không từ nào rỗng trên train (`unused_words_train = 0`). Cụm lớn nhất gấp ~3–5 lần cụm trung vị (k=100: 9845 so với 2891; k=300: 4522 so với 935), tức vài từ rất phổ biến; nhiều khả năng là vùng phẳng/nền. Những từ phổ biến ở cả 5 lớp sẽ bị bước **stop word thị giác** (>80% ảnh train) loại ở bước giao dịch.
- Cụm nhỏ nhất vẫn có 169 keypoint (k=300) nên mọi từ đều có dữ liệu để thành item.
- Inertia giảm từ k=100 sang k=300 (1,18e10 → 1,04e10) vì nhiều cụm hơn, không so sánh trực tiếp để chọn k: k được chọn qua kết quả phân loại.

## 4. Giao dịch (transactions)

Mỗi ảnh thành một giao dịch: tập các từ thị giác "đủ bền" của ảnh đó. Item SPMF = từ + 1 (SPMF cần số dương); id giữ ổn định, không đánh số lại sau khi lọc.

- **Ngưỡng τ = 2:** chỉ giữ từ xuất hiện ít nhất 2 lần trong ảnh. Một từ chỉ xuất hiện đúng 1 lần thường là nhiễu lượng tử hóa của k-means; xuất hiện lặp lại mới cho thấy một cấu trúc hình ảnh thật.
- **Stop word thị giác:** bỏ item có mặt trong **hơn 80% ảnh train** (chỉ đếm trên train, rồi áp cho cả test). Độ phổ biến tính *sau* khi lọc τ, vì đó mới là support mà FPClose nhìn thấy. Với 5 lớp cân bằng, vượt 80% nghĩa là có mặt ở cả 5 lớp nên không phân biệt được lớp nào.
- **Hai chế độ, hai thư mục riêng** (`k{k}_word`, `k{k}_grid`):
  - `word` (**mặc định**): item = từ.
  - `grid`: item = (từ, vùng của lưới 2×2), `item = từ*4 + vùng + 1`. **τ ở chế độ này áp lên cặp (từ, vùng), không phải từ**, nên là một chế độ khác `word`: cùng τ nhưng một cặp ít khi lặp lại đủ τ lần hơn một từ, giao dịch ngắn đi nhiều. Khi báo cáo phải ghi rõ, và không so τ giữa hai chế độ như cùng một đại lượng.
- **Ảnh rỗng** (sau lọc không còn item nào) bị **loại khỏi file .txt**, vì `read_spmf` của FPClose bỏ dòng trống nên số dòng sẽ lệch so với thứ tự ảnh. Ảnh đó được đánh dấu `empty=True` trong `transactions_index.csv` (`line_common`, `line_class` để trống). Số giao dịch *n* trong `ceil(minsup × n)` là số dòng không rỗng của file. Ở cấu hình mặc định hiện không có ảnh rỗng nào (xem bảng bên dưới).
- **File xuất:** mỗi lớp một file (chỉ train), `all_train.txt` (file chung = toàn bộ train, cần để tính support ở các lớp khác cho growth rate), `test.txt` (chỉ để kiểm tra ảnh test có chứa mẫu hay không), `items.csv` (item ↔ từ ↔ vùng, số ảnh chứa, có là stop word không).

### Chọn τ: tiêu chí đặt *trước* khi xem kết quả

Tiêu chí được viết vào docstring của `s05_transactions.py` và commit (`a541b25`) **trước khi chạy script**:

> Với từng (k, chế độ), cố định stop word 80%. Gọi A* là accuracy CV 5-fold (trên train) cao nhất trong τ ∈ {1, 2, 3}. Chọn **τ lớn nhất** có accuracy ≥ A* − 0,01.

Lý do: τ lớn hơn cho giao dịch ngắn và thưa hơn, FPClose chạy nhanh hơn và bớt nhiễu; chênh lệch dưới 1 điểm % không đủ để ưu tiên τ nhỏ. Accuracy lấy từ `LogisticRegression(random_state=42)` trên vector 0/1 "ảnh có item này hay không", `StratifiedKFold(5, shuffle=True, random_state=42)`, **chỉ trên train**. Nếu τ được chọn khác τ mặc định thì chỉ **báo**, không tự đổi cấu hình. Giới hạn: ngưỡng stop word tính trên toàn bộ train trước khi chia fold (không dùng nhãn, nhưng không lồng trong fold).

In [ ]:
run_step("s05_transactions.py")

In [ ]:
tx = pd.read_csv(config.OUT / "transaction_stats.csv")
tx.drop(columns=["tau", "stop"]).set_index(["k", "variant", "group"])

In [ ]:
sens = pd.read_csv(config.OUT / "tau_sensitivity.csv")
display(sens[sens.stop == config.STOP_DF].drop(columns="stop").set_index(["k", "variant", "tau"]))
display(pd.read_csv(config.OUT / "tau_recommendation.csv"))
sens.pivot_table(index=["k", "variant", "tau"], columns="stop", values=["n_stop_items", "cv_acc"])

**Nhận xét (kết quả lần chạy, τ = 2, stop word 80%):**

- Cả 4 cấu hình đều **không có ảnh rỗng**: 140 giao dịch mỗi lớp, 700 train, 300 test.
- **Mật độ cao ở k = 100 `word`:** độ dài TB 63,3 trên 90 item, mật độ 0,70 (Motorbikes 0,83, airplanes 0,54). Giao dịch gần "đầy" nên FPClose sẽ nặng ngay cả ở minsup 60%; cần thử minsup cao trước. Ở k = 300 `word` mật độ 0,34; ở `grid` chỉ 0,26 (k=100) và 0,07 (k=300).
- **airplanes ngắn hơn hẳn** ở mọi cấu hình (k=100: 48,3 so với 58–75 của lớp khác; k=300: 63,8 so với 106–123), hệ quả của ít keypoint (nền trời trơn). Cùng một minsup phần trăm thì support tuyệt đối khác nhau giữa các lớp.
- **Stop word 80% gần như không có tác dụng ở cấu hình mặc định:** chỉ loại 10/100 item ở k=100 `word`, và **0 item** ở k=300 và ở `grid`. Chỉ ở τ = 1 nó mới loại nhiều (84/100 từ ở k=100). Nên khi báo cáo, bước này nên được mô tả đúng là có hiệu lực chủ yếu ở k nhỏ và τ nhỏ, chứ không phải bước lọc mạnh.
- **Kết quả theo tiêu chí chọn τ** (bảng `tau_recommendation` ở trên): k=100 `word` → τ=3, k=100 `grid` → τ=3, k=300 `word` → τ=3, k=300 `grid` → τ=1. **Cả 4 đều khác τ mặc định = 2**, nên theo quy ước ở trên chỉ ghi nhận, chưa đổi cấu hình; quyết định thuộc về nhóm (xem lưu ý dưới).
- Lưu ý khi đọc bảng accuracy: độ lệch chuẩn giữa các fold khoảng 1–3 điểm %, bằng hoặc hơn nhiều chênh lệch giữa các τ (ví dụ k=100 `grid`: 0,876 / 0,884 / 0,889). Accuracy này đo trên *mọi item*, không phải trên các mẫu khai thác, nên chỉ là chỉ báo gián tiếp cho việc chọn τ. k=300 `grid` tụt mạnh ở τ=3 (0,831) vì giao dịch chỉ còn ~33 item trên 1200 item, thưa quá.